# CityLens Baseline Comparison

Use this notebook for the next experiment phase only:

- lock the current `prithvi_rgb_lora` reference results
- run fair satellite backbone comparisons
- summarize metrics into one table
- decide whether street-only and fusion are worth running next

This notebook is intentionally smaller than `colab_citylens_full.ipynb` and calls the existing training entrypoints instead of duplicating the full pipeline.

## Colab setup

Upload this notebook to a fresh Colab runtime, choose a GPU, then run all cells. This notebook mounts Drive, clones the repo, installs the learned-model dependencies, and runs only the comparison-phase experiments.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_ROOT = Path('/content/CityLens')
if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/tsinghua-fib-lab/CityLens.git', str(REPO_ROOT)], check=True)
print('Repo root:', REPO_ROOT)

In [ ]:
# Install learned-model dependencies.
MARKER = Path('/content/.citylens_compare_deps_ready_v1')
if not MARKER.exists():
    cmd = [
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', '--no-cache-dir',
        'numpy==1.26.4',
        'scipy==1.13.1',
        'pandas==2.2.2',
        'scikit-learn==1.5.2',
        'terratorch',
        'timm',
        'peft',
        'captum',
        'safetensors',
    ]
    subprocess.check_call(cmd)
    MARKER.write_text('ok', encoding='utf-8')
    print('Dependencies installed. Restarting runtime once...')
    os.kill(os.getpid(), 9)
else:
    print('Dependencies already prepared for this runtime.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = Path('/content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data')
os.environ['CITYLENS_DATA_ROOT'] = str(DATA_ROOT)
print('CITYLENS_DATA_ROOT =', os.environ['CITYLENS_DATA_ROOT'])

In [ ]:
required = [
    DATA_ROOT / 'Benchmark',
    DATA_ROOT / 'Results',
    REPO_ROOT / 'CityLens',
]
for path in required:
    print(path, 'exists=' + str(path.exists()))
if not (DATA_ROOT / 'Benchmark').exists():
    raise SystemExit('CityLens data not found in Drive. Reuse the previous full notebook once to prepare the dataset first.')

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import pandas as pd

DATA_ROOT = Path('/content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data')
CITYLENS_ROOT = Path('/content/CityLens')
RESULTS_ROOT = DATA_ROOT / 'Results' / 'global_learned'

REFERENCE_MODELS = ['prithvi_rgb_lora', 'dinov2_sat', 'resnet50_sat']
TASK_EPOCHS = {
    'gdp': 20,
    'acc2health': 30,
    'build_height': 30,
    'pop': 5,
}

LOCKED_PRITHVI = {
    'gdp': {'r2': 0.5807730555534363, 'rmse': 331463584.0, 'best_epoch': 14, 'epochs': 20},
    'acc2health': {'r2': 0.39014101028442383, 'rmse': 9.55018424987793, 'best_epoch': 9, 'epochs': 30},
    'build_height': {'r2': 0.8681523203849792, 'rmse': 2.534539222717285, 'best_epoch': 9, 'epochs': 30},
    'pop': {'r2': -0.03239285945892334, 'rmse': 21641.24609375, 'best_epoch': 2, 'epochs': 5},
}

COMMON = {
    'branch': 'satellite',
    'street_model': 'clip_vitb16',
    'batch_size': 8,
    'image_size': 224,
    'lr': 2e-4,
    'backbone_lr': 2e-4,
    'head_lr': 1e-3,
    'weight_decay': 1e-2,
    'seed': 42,
    'lora_r': 8,
    'target_transform': 'log1p',
    'val_frac': 0.1,
    'num_workers': 2,
}

assert DATA_ROOT.exists(), f'Missing data root: {DATA_ROOT}'
assert CITYLENS_ROOT.exists(), f'Missing repo root: {CITYLENS_ROOT}'
print('Results root:', RESULTS_ROOT)

In [ ]:
def run_train(task_name: str, satellite_model: str, epochs: int, resume: bool = True, skip_if_done: bool = True):
    cmd = [
        'python', '-m', 'evaluate.global_learned.train',
        '--task_name', task_name,
        '--branch', COMMON['branch'],
        '--satellite_model', satellite_model,
        '--street_model', COMMON['street_model'],
        '--epochs', str(epochs),
        '--batch_size', str(COMMON['batch_size']),
        '--image_size', str(COMMON['image_size']),
        '--lr', str(COMMON['lr']),
        '--backbone_lr', str(COMMON['backbone_lr']),
        '--head_lr', str(COMMON['head_lr']),
        '--weight_decay', str(COMMON['weight_decay']),
        '--seed', str(COMMON['seed']),
        '--lora_r', str(COMMON['lora_r']),
        '--target_transform', COMMON['target_transform'],
        '--val_frac', str(COMMON['val_frac']),
        '--num_workers', str(COMMON['num_workers']),
        '--data_root', str(DATA_ROOT),
    ]
    if resume:
        cmd.append('--resume')
    if skip_if_done:
        cmd.append('--skip_if_done')

    print('\n$', ' '.join(cmd))
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    subprocess.run(cmd, cwd=str(CITYLENS_ROOT), env=env, check=True)


def run_satellite_matrix(tasks, models):
    for task_name in tasks:
        epochs = TASK_EPOCHS[task_name]
        for satellite_model in models:
            run_train(task_name, satellite_model, epochs)


## Run matched satellite baselines

Recommended immediate comparison set:

1. `build_height`: `dinov2_sat`, `resnet50_sat`
2. `gdp`: `dinov2_sat`, `resnet50_sat`
3. `acc2health`: `dinov2_sat`, `resnet50_sat`

Leave `pop` optional unless you want a fairer failure-case comparison.

In [ ]:
TASKS_TO_RUN = ['build_height', 'gdp', 'acc2health']
SAT_MODELS_TO_RUN = ['dinov2_sat', 'resnet50_sat']

# Set EXECUTE = True when ready.
EXECUTE = False

if EXECUTE:
    run_satellite_matrix(TASKS_TO_RUN, SAT_MODELS_TO_RUN)
else:
    for task_name in TASKS_TO_RUN:
        for satellite_model in SAT_MODELS_TO_RUN:
            print(task_name, satellite_model, TASK_EPOCHS[task_name])


In [ ]:
def find_metrics(task_name: str, satellite_model: str, epochs: int):
    task_root = RESULTS_ROOT / task_name
    if not task_root.exists():
        return None
    pattern = f"{task_name}-satellite-{satellite_model}-clip_vitb16-bs8-ep{epochs}-lr0.0002-blr0.0002-hlr0.001-ttlog1p-seed42"
    exp_dir = task_root / pattern
    metrics_path = exp_dir / 'metrics.json'
    if not metrics_path.exists():
        return None
    payload = json.loads(metrics_path.read_text())
    return {
        'task': task_name,
        'model': satellite_model,
        'epochs': epochs,
        'best_epoch': payload.get('best_epoch'),
        'r2': payload.get('r2'),
        'rmse': payload.get('rmse'),
        'mae': payload.get('mae'),
        'mse': payload.get('mse'),
        'path': str(exp_dir),
    }


rows = []
for task_name, ref in LOCKED_PRITHVI.items():
    rows.append({
        'task': task_name,
        'model': 'prithvi_rgb_lora',
        'epochs': ref['epochs'],
        'best_epoch': ref['best_epoch'],
        'r2': ref['r2'],
        'rmse': ref['rmse'],
        'mae': None,
        'mse': None,
        'path': 'docs/PRITHVI_SATELLITE_REFERENCE.md',
    })

for task_name in TASK_EPOCHS:
    for satellite_model in ['dinov2_sat', 'resnet50_sat']:
        row = find_metrics(task_name, satellite_model, TASK_EPOCHS[task_name])
        if row is not None:
            rows.append(row)

df = pd.DataFrame(rows)
df.sort_values(['task', 'model']) if not df.empty else df

In [ ]:
if df.empty:
    print('No baseline comparison rows found yet.')
else:
    pivot = df.pivot_table(index='task', columns='model', values='r2', aggfunc='first')
    display(pivot)

    wins = 0
    considered = 0
    for task_name in ['build_height', 'gdp', 'acc2health']:
        if task_name not in pivot.index:
            continue
        if 'prithvi_rgb_lora' not in pivot.columns:
            continue
        other_values = [pivot.loc[task_name, c] for c in pivot.columns if c != 'prithvi_rgb_lora' and pd.notna(pivot.loc[task_name, c])]
        if not other_values:
            continue
        considered += 1
        if pivot.loc[task_name, 'prithvi_rgb_lora'] > max(other_values):
            wins += 1

    if considered == 0:
        print('Need baseline runs before making the modality decision.')
    elif wins >= 2:
        print('Recommendation: proceed to street-only clip_vitb16 and late fusion next.')
    else:
        print('Recommendation: stop and frame this as a smaller satellite empirical study unless stronger baseline results arrive.')
